# Set-up

In [1]:
### Imports ###
import pandas as pd
import pickle
from skfda import FDataGrid
from datetime import date, datetime, timedelta
import numpy as np

from src.preprocessing import GMEPreprocessor, ExogPreprocessor
from src.curves import SupplyDemandTimeSeries
from src.forecasting import LassoVARX, SupplyDemandForecaster
from src.utils import fix_daylight_saving_time

In [ ]:
### Parameters ###

# Input paths
bids_path = 'data/source/MGPDomandaOfferta/MGPDomandaOfferta.pkl'
coupling_path = 'data/source/MGPMarketCoupling/balance_coupling.csv'
prices_path = 'data/source/MGPPrezzi/mgp_prices.pkl'
exog_path = 'data/source/predictors.pkl'

# Processed paths
curves_path = 'data/processed/sdts.pkl'

# Output paths
pred_curves_path = 'data/output/sdts_pred_aic_wo_ntc.pkl'
pred_prices_path = 'data/output/prices_pred_aic_wo_ntc.pkl'

# Whether to re-preprocess curves (long!)
rerun_curves_preprocessing = False

# Model parameters

K_supply = 5
K_demand = 3

exog_variables = [
    'GFSo Solar ITA',
    'ECo Wind ITA',
    'Load_IT',
    'FR > IT',
    'IT > FR',
    'CH > IT',
    'IT > CH',
    'AT > IT',
    'IT > AT',
    'SI > IT',
    'IT > SI',
    'IT > ME',
    'ME > IT',
    'IT > GR',
    'GR > IT',
]

# Calibration window and testing period
calibration_window = timedelta(days=358)
test_start_date = date(2024, 1, 1)
test_end_date = date(2024, 12, 31)
print(f"Calibration_window: {calibration_window.days} days")
print(f"Testing period: {test_start_date} to {test_end_date}")

Calibration_window: 358 days
Testing period: 2024-01-01 to 2024-12-31


In [5]:
### Read data ###

bids = pd.read_pickle(bids_path)
coupling = pd.read_csv(coupling_path)
prices = pd.read_pickle(prices_path)
exog = pd.read_pickle(exog_path)

In [6]:
### Preprocessing ###

# Start and end datetimes
test_start = pd.Timestamp(test_start_date) # Time information automatically set at 00:00:00
test_end = pd.Timestamp(test_end_date) + timedelta(hours=23) # Time information set to 23:00:00
train_start = test_start - calibration_window
preprocess_start = train_start - timedelta(weeks=1) # We need one week of past data to compute the lags

# Curves
if rerun_curves_preprocessing:
    preprocessor = GMEPreprocessor()
    data_matrix_off, grid_points = preprocessor.get_curves_dataset(bids, type='OFF', balance_df=coupling)
    data_matrix_bid, _ = preprocessor.get_curves_dataset(bids, type='BID', balance_df=coupling)

    sd = SupplyDemandTimeSeries(
        FDataGrid(data_matrix_off, grid_points, sample_names=preprocessor.timestamps, extrapolation='bounds'),
        FDataGrid(data_matrix_bid, grid_points, sample_names=preprocessor.timestamps, extrapolation='bounds')
    )
    sd.to_pickle(curves_path)
else:
    with open(curves_path, 'rb') as file:
        sd = pickle.load(file)
sd = sd[preprocess_start:test_end]

# Exog
exogprep = ExogPreprocessor(
    start_date=preprocess_start.date(),
    end_date=test_end_date,
    exog_variables=exog_variables
)
exog = exogprep.preprocess_exog(exog)

# Prices
prices = fix_daylight_saving_time(prices)
prices_true = prices.loc[test_start:test_end, 'NAT']

In [7]:
### Forecasting ###

model = LassoVARX(ar_structure='concurrent', var_structure='concurrent',
                    calibration_window=calibration_window, criterion='aic', n_jobs=1)

forecaster = SupplyDemandForecaster(model, exogprep, K_supply=K_supply, K_demand=K_demand)
sd_pred = forecaster.fit_forecast_daily_recal(sd, exog, test_start=date(2024, 1, 1))
prices_pred = sd_pred.get_clearing_prices()

Daily Recalibration Progress:   0%|          | 0/366 [00:00<?, ?it/s]/Users/guillaume/Projects/.venvs/moc_forecast/lib/python3.11/site-packages/sklearn/base.py:1365: UserWarning: With alpha=0, this algorithm does not converge well. You are advised to use the LinearRegression estimator
  return fit_method(estimator, *args, **kwargs)
/Users/guillaume/Projects/.venvs/moc_forecast/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:695: UserWarning: Coordinate descent with no regularization may lead to unexpected results and is discouraged.
  model = cd_fast.enet_coordinate_descent(
/Users/guillaume/Projects/.venvs/moc_forecast/lib/python3.11/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 9.708e+00, tolerance: 2.549e-02
Linear regression models with a zero l1 penalization

In [8]:
### Evaluate and save ###

sd_pred.to_pickle(pred_curves_path)
prices_pred.to_pickle(pred_prices_path)
print("MAE: {:.2f}€/MWh".format((prices_true - prices_pred).abs().mean()))

MAE: 8.63€/MWh


<HR>

# Tests

In [7]:
model = LassoVARX(ar_structure='concurrent', var_structure='concurrent',
                    calibration_window=calibration_window, criterion='bic', n_jobs=1)
forecaster = SupplyDemandForecaster(model, exogprep, K_supply=K_supply, K_demand=K_demand)
endog = forecaster._transform_endog(sd)
exog_transformed = forecaster._transform_exog(exog, forecaster.preprocessor.dummy_columns)

In [8]:
model.fit_forecast(endog, exog_transformed, test_start_date)

2025-09-25 18:06:16,270 - INFO - Training period is from 2023-01-08 to 2023-12-31
2025-09-25 18:06:16,930 - INFO - Forecasting period is from 2024-01-01 to 2024-12-31


,FPC1o,FPC2o,FPC3o,FPC4o,FPC5o,FPC1b,FPC2b,FPC3b
2024-01-01 00:00:00,-1.137209,2.021604,-0.311896,0.276881,0.092595,-1.698447,-0.737274,-0.133041
2024-01-01 01:00:00,-1.084138,2.176792,-0.354792,0.294119,0.104510,-1.814943,-0.927108,-0.147736
2024-01-01 02:00:00,-1.192219,2.125198,-0.235702,0.250969,0.035598,-1.972090,-1.084522,-0.114275
2024-01-01 03:00:00,-1.224239,2.129900,-0.232531,0.336341,0.077955,-2.126911,-1.063427,-0.173709
2024-01-01 04:00:00,-1.244533,2.161937,-0.217592,0.386228,0.213920,-2.110360,-0.981249,-0.141321
...,...,...,...,...,...,...,...,...
2024-12-31 19:00:00,-0.850338,1.170386,-0.084398,-0.808104,1.956873,0.279186,-0.100977,0.943531
2024-12-31 20:00:00,-0.960772,1.287590,-0.219145,-1.054383,1.784239,-0.095033,-0.080709,0.906777
2024-12-31 21:00:00,-1.087656,1.417553,-0.147605,-1.206849,1.785745,-0.448066,0.057299,1.112895
2024-12-31 22:00:00,-1.223876,1.479479,-0.177058,-1.153531,1.412955,-0.752940,0.136875,0.933038


In [9]:
Ys, Xs = model._build_XY(endog, exog_transformed)

In [10]:
endog

,FPC1o,FPC2o,FPC3o,FPC4o,FPC5o,FPC1b,FPC2b,FPC3b
2023-01-01 00:00:00,-3.322868,-1.002567,2.016078,1.851720,-1.907379,-1.596407,-1.863054,-0.143802
2023-01-01 01:00:00,-3.328844,-0.998738,2.040334,1.827332,-1.908939,-1.791509,-1.937868,-0.216853
2023-01-01 02:00:00,-3.327533,-0.998550,2.076669,1.773064,-1.983693,-1.942827,-1.897682,-0.115136
2023-01-01 03:00:00,-3.377727,-0.990177,2.105002,1.736669,-2.006380,-2.042394,-1.939930,-0.077954
2023-01-01 04:00:00,-3.390146,-0.999390,2.112198,1.725188,-2.060628,-2.072726,-1.915918,-0.028048
...,...,...,...,...,...,...,...,...
2024-12-31 19:00:00,-0.757946,1.387442,0.278584,-1.872336,3.168781,0.335177,0.114663,0.901054
2024-12-31 20:00:00,-0.858081,1.233503,0.396761,-1.757319,2.645912,0.056593,0.583438,0.878477
2024-12-31 21:00:00,-0.901846,1.261147,0.459751,-2.171178,2.498313,-0.303964,-0.038580,0.552786
2024-12-31 22:00:00,-1.164687,1.457419,-0.171244,-2.047738,2.465139,-0.669690,0.045793,0.776406


In [11]:
Ys['FPC1o']

,0,1,2,3,4,5,6,7,8,9,...,14,15,16,17,18,19,20,21,22,23
2023-01-08,-2.428327,-2.613018,-2.737172,-2.777245,-2.770829,-2.727794,-2.603073,-2.435978,-2.182250,-1.848043,...,-1.871473,-2.011676,-1.821420,-1.517438,-1.564583,-1.565380,-1.586822,-1.520829,-1.694452,-2.077068
2023-01-09,-2.504346,-2.587597,-2.556255,-2.523187,-2.433128,-2.165363,-1.404209,-1.032523,-0.728279,-0.492211,...,-0.095179,-0.204449,-0.097344,0.024961,0.157477,0.118948,-0.054744,-0.207880,-0.609522,-0.821028
2023-01-10,-1.458405,-1.680562,-1.787839,-1.842994,-1.859126,-1.575736,-1.279851,-0.788999,-0.584433,-0.292575,...,0.115868,-0.014685,-0.095492,-0.168621,-0.040785,-0.126253,-0.224117,-0.578117,-0.883539,-1.074334
2023-01-11,-1.536948,-1.700998,-1.827208,-1.878409,-1.824927,-1.608751,-1.326916,-0.974820,-0.662993,-0.302063,...,-0.310200,-0.681598,-0.932404,-0.868791,-0.743681,-0.782609,-0.908363,-1.185732,-1.392739,-1.474361
2023-01-12,-1.532478,-1.634798,-1.613997,-1.632439,-1.593987,-1.533044,-1.355877,-1.089315,-0.557741,-0.365561,...,-0.295140,-0.399694,-0.562642,-0.522247,-0.392249,-0.452887,-0.596964,-0.760560,-1.170159,-1.260284
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-27,-1.178852,-1.189992,-1.226340,-1.208940,-1.203279,-1.112424,-0.921339,-0.558826,-0.345936,-0.100143,...,-0.147858,-0.476433,-0.414535,-0.322809,-0.115265,-0.132506,-0.346751,-0.600045,-0.809990,-0.864665
2024-12-28,-1.356142,-1.444618,-1.312387,-1.316668,-1.242248,-1.248028,-0.990669,-0.805094,-0.508622,-0.456571,...,-0.301234,-0.643523,-0.735855,-0.645416,-0.580363,-0.615918,-0.746990,-0.808887,-0.998178,-1.105804
2024-12-29,-1.816257,-1.841282,-1.846574,-1.946437,-1.922763,-1.861487,-1.790145,-1.770331,-1.423790,-1.070592,...,-0.954500,-1.220176,-1.372556,-1.371827,-1.208351,-1.138712,-1.295313,-1.322387,-1.462436,-1.595169
2024-12-30,-1.787357,-1.822639,-1.857730,-1.808966,-1.801174,-1.707287,-1.587160,-1.242786,-0.730772,-0.338143,...,-0.271558,-0.673846,-0.880806,-0.933321,-0.858627,-0.924332,-0.897832,-1.052144,-1.343675,-1.398939


In [12]:
Xs['FPC1b'][0]

,Load_IT,FR > IT,IT > FR,CH > IT,IT > CH,AT > IT,IT > AT,SI > IT,IT > SI,IT > ME,...,FPC3o_h0_L1,FPC3o_h0_L7,FPC4o_h0_L1,FPC4o_h0_L7,FPC5o_h0_L1,FPC5o_h0_L7,FPC2b_h0_L1,FPC2b_h0_L7,FPC3b_h0_L1,FPC3b_h0_L7
2023-01-08,-1.381040,0.313359,-0.861623,1.075334,1.070801,-0.066025,0.344924,0.350165,0.412839,0.238286,...,1.791258,2.016078,1.629316,1.851720,-1.602562,-1.907379,0.416078,-1.863054,2.667238,-0.143802
2023-01-09,-1.526998,1.020449,-0.861623,-0.982912,1.070801,0.224814,0.344924,1.088376,0.412839,0.238286,...,1.688740,1.960242,1.984440,1.852609,-1.570945,-1.636211,-0.526471,-1.847418,1.799927,-0.102721
2023-01-10,-0.846195,0.281219,-0.861623,0.932521,1.070801,-0.080567,0.344924,0.317501,0.412839,0.238286,...,1.673809,1.824238,1.824546,2.015773,-1.853810,-1.246435,-1.109673,0.879311,0.955885,2.022882
2023-01-11,-0.798923,0.864889,-0.861623,1.424189,1.070801,0.159375,0.344924,0.925055,0.412839,0.238286,...,1.452514,1.678855,1.930928,1.771256,-0.686405,-1.034148,0.392995,-0.006142,2.574291,1.215961
2023-01-12,-0.700952,0.925313,-0.861623,1.742520,1.070801,0.181188,0.344924,0.990384,0.412839,0.238286,...,1.259713,1.306196,2.318110,1.641674,-1.135863,-1.114389,0.048263,-0.497413,1.751420,1.388753
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-27,-1.478155,1.264716,1.429699,-0.731083,1.070801,0.246627,0.433447,0.147647,0.412839,0.238286,...,-1.246977,-1.805544,-0.427901,0.109789,3.378728,1.853707,-0.062267,-0.529261,0.248361,-0.263100
2024-12-28,-1.337481,1.710825,1.429699,-0.731083,1.070801,0.472027,0.433447,0.493887,0.412839,0.238286,...,-1.224668,-1.864525,-0.374209,0.251412,3.185689,2.823984,0.479746,1.815778,1.533985,1.140418
2024-12-29,-1.401034,2.240499,1.429699,-1.005806,1.070801,0.537466,0.433447,0.911990,0.412839,0.238286,...,-0.953255,-1.818902,-0.892851,-0.204425,2.435181,3.002736,0.874670,0.710751,1.854992,1.382338
2024-12-30,-1.477155,1.829102,1.429699,-0.731083,1.070801,0.537466,0.433447,0.591880,0.412839,0.238286,...,-0.350449,-1.811043,-0.398802,0.029770,1.888894,2.874290,1.305383,-0.309004,1.762227,0.425969


In [13]:
index_test = endog.index.date >= test_start_date - timedelta(weeks=1)
index_train = (endog.index.date < test_start_date) & (endog.index.date >= test_start_date - timedelta(weeks=1) - model.calibration_window)

Ys_train = {pc: target[target.index < test_start_date] for pc, target in Ys.items()}
Ys_test = {pc: target[test_start_date:] for pc, target in Ys.items()}

Xs_train = {pc: {h: features[features.index < test_start_date] for h, features in target.items()} for pc, target in Ys.items()}
Xs_test = {pc: {h: features[test_start_date:] for h, features in target.items()} for pc, target in Ys.items()}

In [14]:
Xs_test['FPC1b'][0].index[0]

datetime.date(2024, 1, 1)

In [ ]:
import cProfile, pstats

cProfile.run('model.fit_forecast(endog, exog_transformed, test_start_date)', 'profile')
p = pstats.Stats('profile')
p.sort_stats('cumtime').print_stats(15)

2025-09-25 18:06:20,347 - INFO - Training period is from 2023-01-08 to 2023-12-31
2025-09-25 18:06:21,764 - INFO - Forecasting period is from 2024-01-01 to 2024-12-31


Thu Sep 25 18:06:23 2025    profile

         7852008 function calls (7692885 primitive calls) in 4.719 seconds

   Ordered by: cumulative time
   List reduced from 1184 to 15 due to restriction <15>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    4.722    4.722 {built-in method builtins.exec}
        1    0.004    0.004    4.722    4.722 <string>:1(<module>)
        1    0.001    0.001    4.719    4.719 /Users/guillaume/Projects/moc_forecast/src/forecasting.py:229(fit_forecast)
        2    0.029    0.014    2.823    1.412 /Users/guillaume/Projects/moc_forecast/src/forecasting.py:56(_build_XY)
        1    0.002    0.002    1.490    1.490 /Users/guillaume/Projects/moc_forecast/src/forecasting.py:157(fit)
  576/384    0.003    0.000    1.462    0.004 /Users/guillaume/Projects/.venvs/moc_forecast/lib/python3.11/site-packages/sklearn/base.py:1348(wrapper)
     4436    0.013    0.000    1.218    0.000 /Users/guillaume/Projects/.venvs

In [21]:
import cProfile, pstats

cProfile.run('model._fit_forecast_from_XY(Ys, Xs, test_start_date)', 'profile')
p = pstats.Stats('profile')
p.sort_stats('cumtime').print_stats(15)

2025-09-25 18:08:14,913 - INFO - Training period is from 2023-01-08 to 2023-12-31
2025-09-25 18:08:14,914 - INFO - Forecasting period is from 2024-01-01 to 2024-12-31


Thu Sep 25 18:08:16 2025    profile

         2850935 function calls (2812476 primitive calls) in 2.048 seconds

   Ordered by: cumulative time
   List reduced from 1014 to 15 due to restriction <15>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    2.052    2.052 {built-in method builtins.exec}
        1    0.002    0.002    2.052    2.052 <string>:1(<module>)
        1    0.000    0.000    2.050    2.050 /Users/guillaume/Projects/moc_forecast/src/forecasting.py:264(_fit_forecast_from_XY)
        1    0.003    0.003    1.534    1.534 /Users/guillaume/Projects/moc_forecast/src/forecasting.py:157(fit)
  576/384    0.003    0.000    1.502    0.004 /Users/guillaume/Projects/.venvs/moc_forecast/lib/python3.11/site-packages/sklearn/base.py:1348(wrapper)
      192    0.041    0.000    1.032    0.005 /Users/guillaume/Projects/.venvs/moc_forecast/lib/python3.11/site-packages/sklearn/linear_model/_least_angle.py:2221(fit)
     2880    0.056 

In [23]:
exog

,Load_IT,FR > IT,IT > FR,CH > IT,IT > CH,AT > IT,IT > AT,SI > IT,IT > SI,IT > ME,ME > IT,IT > GR,GR > IT,RES,is_Holiday,is_Monday,is_Saturday
2023-01-01 00:00:00,20762.0,3738.0,1660.0,1411.0,1910.0,305.0,145.0,641.0,680.0,600.0,600.0,500.0,500.0,414.4651,True,False,False
2023-01-01 01:00:00,19220.0,3738.0,1660.0,1411.0,1910.0,305.0,145.0,641.0,680.0,600.0,600.0,500.0,500.0,434.8042,True,False,False
2023-01-01 02:00:00,18280.0,3738.0,1660.0,1411.0,1910.0,305.0,145.0,641.0,680.0,600.0,600.0,500.0,500.0,416.6655,True,False,False
2023-01-01 03:00:00,17553.0,3738.0,1660.0,1271.0,1910.0,305.0,145.0,641.0,680.0,600.0,600.0,500.0,500.0,413.5910,True,False,False
2023-01-01 04:00:00,17670.0,3738.0,1660.0,1177.0,1910.0,305.0,145.0,641.0,680.0,600.0,600.0,500.0,500.0,418.7922,True,False,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-12-31 19:00:00,32798.0,4241.0,1945.0,2571.0,1765.0,386.0,131.0,357.0,757.0,600.0,600.0,500.0,500.0,271.8840,False,False,False
2024-12-31 20:00:00,30599.0,4241.0,1945.0,2571.0,1765.0,386.0,131.0,357.0,757.0,600.0,600.0,500.0,500.0,284.8980,False,False,False
2024-12-31 21:00:00,28231.0,3757.0,1959.0,2572.0,1777.0,341.0,132.0,312.0,737.0,600.0,600.0,500.0,500.0,307.8530,False,False,False
2024-12-31 22:00:00,26359.0,3757.0,1959.0,2572.0,1777.0,341.0,132.0,312.0,737.0,600.0,600.0,500.0,500.0,340.5530,False,False,False


In [22]:
cProfile.run('model.fit_forecast(endog, exog, test_start_date)', 'profile')
p = pstats.Stats('profile')
p.sort_stats('cumtime').print_stats(15)

2025-09-26 09:59:56,758 - INFO - Training period is from 2023-01-08 to 2023-12-31
2025-09-26 09:59:58,248 - INFO - Forecasting period is from 2024-01-01 to 2024-12-31
/Users/guillaume/Projects/.venvs/moc_forecast/lib/python3.11/site-packages/sklearn/linear_model/_least_angle.py:723: ConvergenceWarning: Regressors in active set degenerate. Dropping a regressor, after 14 iterations, i.e. alpha=4.232e-01, with an active set of 12 regressors, and the smallest cholesky pivot element being 2.220e-16. Reduce max_iter or increase eps parameters.
  warnings.warn(
/Users/guillaume/Projects/.venvs/moc_forecast/lib/python3.11/site-packages/sklearn/linear_model/_least_angle.py:753: ConvergenceWarning: Early stopping the lars path, as the residues are small and the current value of alpha is no longer well controlled. 34 iterations, alpha=7.655e-01, previous alpha=3.790e-03, with an active set of 29 regressors.
  warnings.warn(
/Users/guillaume/Projects/.venvs/moc_forecast/lib/python3.11/site-package

Fri Sep 26 10:00:00 2025    profile

         7753619 function calls (7594496 primitive calls) in 4.987 seconds

   Ordered by: cumulative time
   List reduced from 1211 to 15 due to restriction <15>

   ncalls  tottime  percall  cumtime  percall filename:lineno(function)
        1    0.000    0.000    4.991    4.991 {built-in method builtins.exec}
        1    0.004    0.004    4.990    4.990 <string>:1(<module>)
        1    0.012    0.012    4.986    4.986 /Users/guillaume/Projects/moc_forecast/src/forecasting.py:229(fit_forecast)
        2    0.032    0.016    2.928    1.464 /Users/guillaume/Projects/moc_forecast/src/forecasting.py:56(_build_XY)
        1    0.003    0.003    1.595    1.595 /Users/guillaume/Projects/moc_forecast/src/forecasting.py:157(fit)
  576/384    0.003    0.000    1.562    0.004 /Users/guillaume/Projects/.venvs/moc_forecast/lib/python3.11/site-packages/sklearn/base.py:1348(wrapper)
     4436    0.013    0.000    1.263    0.000 /Users/guillaume/Projects/.venvs

In [16]:
sd_naive = sd[test_start - timedelta(days=7):].get_naive_forecast()
prices_naive = sd_naive.get_clearing_prices()
print("MAE: {:.2f}€/MWh".format((prices_true - prices_naive).abs().mean()))

MAE: 11.34€/MWh
